# Testing with already trained things

In [5]:
from pathlib import Path
import sys
import pandas as pd
from torch.utils.data import DataLoader, Dataset, Subset
import torch.optim as optim
import matplotlib.pyplot as plt
import numpy as np
from tqdm import tqdm

from datetime import datetime

In [ ]:
ticker = "AAPL"
start_date = "2010-01-01"
end_date = "2024-12-31"

# training parameters
sequence_length = 252
validation_ratio = 0.2

execute_train = False
load_outputs = True
checkpoint_path = "../checkpoints/scratch_ckpt_epoch_49_2026-01-12.pt"
num_samples = 100
date_to_load = "2026-01-08"

In [7]:
# 1) go back to parent folder
p1 = Path.cwd().parent

# 2) go back to parent of the parent
p2 = Path.cwd().parent.parent

# 3) enter folder "CSDI" (assumed to be inside that parent-of-parent)
csdi_dir = p2 / "CSDI"

# Make sure it exists (optional but recommended)
if not csdi_dir.is_dir():
    raise FileNotFoundError(f"Folder not found: {csdi_dir}")

# Add CSDI to Python import path so imports work from anywhere
sys.path.insert(0, str(csdi_dir))

dataset_dir = p1 / "data"
config_dir = p1 / "configs"

sys.path.insert(0, str(dataset_dir))

In [ ]:
# 4) imports
import torch
import sp500_data
from csdi_scratch import CSDIFromScratch
from sp500_data import MultiFinancialDataset


In [9]:
import yaml

def load_config(config_path):
    """
    Reads a YAML configuration file and returns a nested dictionary.
    
    Args:
        config_path (str): Path to the .yaml file.
        
    Returns:
        dict: Configuration parameters for CSDI initialization.
    """
    try:
        with open(config_path, "r") as f:
            # safe_load handles standard YAML tags and avoids code injection
            config = yaml.safe_load(f)
        return config
    except FileNotFoundError:
        raise FileNotFoundError(f"Config file not found at {config_path}")
    except yaml.YAMLError as exc:
        raise Exception(f"Error parsing YAML file: {exc}")

# Usage for your thesis project
config = load_config("../configs/base_csdi.yaml")

# Accessing parameters (matches the CSDI_base logic)
target_dim = config["model"]["target_dim"]
beta_start = config["diffusion"]["beta_start"]

print(f"Initialized with target_dim: {target_dim}")

Initialized with target_dim: 5


In [ ]:
# class CSDI_Financial(CSDI_base):
#     def __init__(self, config, device, target_dim=5):
#         super().__init__(target_dim, config, device)

#     def process_data(self, batch):
#         # Maps your FinancialDataset dictionary to the model

#         observed_data = batch["observed_data"].to(self.device).float()
#         observed_mask = batch["observed_mask"].to(self.device).float()
#         observed_tp = batch["timepoints"].to(self.device).float()
#         gt_mask = batch["gt_mask"].to(self.device).float()

#         cond_mask = batch.get("cond_mask", observed_mask).to(self.device).float()

#         cut_length = torch.zeros(len(observed_data)).long().to(self.device)

#         # Ensure (B, K, L) shape
#         return (observed_data, observed_mask, observed_tp, gt_mask, cond_mask, cut_length)

In [11]:
# Device configuration
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

df = pd.read_csv(f'../data/{ticker}_{start_date}_{end_date}_processed.csv')
print(df)

def create_dataloaders(df, seq_len, val_rat, batch_size):
    # 1. Create full dataset
    full_dataset = FinancialDataset(dataset=df, seq_len=seq_len, close_idx=3, forecast_horizon=None)
    print("Full dataset Okay")
    
    # 2. Calculate split point (Chronological)
    total_len = len(full_dataset)
    val_len = int(total_len * val_rat)
    train_len = total_len - val_len
    
    # 3. Create subsets
    # Training gets the earlier data, Validation gets the most recent
    train_dataset = Subset(full_dataset, range(0, train_len))
    print("Train Okay")
    val_dataset = Subset(full_dataset, range(train_len, total_len))
    print("Validation Okay")
    
    # 4. Create Loaders
    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
    print("Train Loader Okay")
    val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)
    print("Validation Loader Okay")
    return train_loader, val_loader

          Open      High       Low     Close    Volume
0     0.183717 -0.232828  0.379857  0.043035  0.635150
1    -0.053535 -0.486570 -0.497403 -0.968573 -0.273358
2     0.264426 -0.418403  0.017449 -0.160763 -0.464357
3    -0.168128 -0.280224  0.140662  0.322050 -0.201814
4     0.279136 -0.423615 -0.478908 -0.560005  0.104901
...        ...       ...       ...       ...       ...
3767  0.041223 -0.442469  0.344968  0.118928 -4.098826
3768  0.020710  0.072896  0.615755  0.594633 -1.801145
3769 -0.056859 -0.234643  0.467274  0.125221  0.509984
3770 -0.450278 -0.874233 -0.899743 -0.814587  1.412766
3771 -1.193710 -1.394892 -0.629791 -0.815810 -0.556674

[3772 rows x 5 columns]


In [12]:
train_loader, val_loader = create_dataloaders(df, sequence_length, validation_ratio, batch_size=config['train']['batch_size'])

Full dataset Okay
Train Okay
Validation Okay
Train Loader Okay
Validation Loader Okay


# Core statistics

In [13]:
print("Descriptive Statistics for Standardized ticker: ", ticker)
print(df.describe())

Descriptive Statistics for Standardized ticker:  AAPL
               Open          High           Low         Close        Volume
count  3.772000e+03  3.772000e+03  3.772000e+03  3.772000e+03  3.772000e+03
mean  -8.928076e-11 -1.447193e-08 -1.117504e-08  8.205493e-09  8.020541e-09
std    1.000132e+00  1.000132e+00  1.000132e+00  1.000132e+00  1.000133e+00
min   -1.201118e+01 -8.145225e+00 -1.564787e+01 -7.898292e+00 -5.660525e+00
25%   -3.607679e-01 -5.361357e-01 -3.761705e-01 -4.784359e-01 -6.256660e-01
50%    1.616051e-02 -1.359673e-01  1.656052e-01  1.827763e-03 -5.402130e-02
75%    3.940182e-01  4.203780e-01  5.450312e-01  5.314068e-01  5.893394e-01
max    8.065092e+00  8.187183e+00  5.700387e+00  6.389341e+00  5.774343e+00


# Training from Scratch

In [14]:
# Initialize your custom model and move to device
model = CSDI_Financial(config, device).to(device)

# Standard optimizer for CSDI
optimizer = optim.Adam(model.parameters(), lr=config["train"]["lr"])

c:\Users\Lenovo\anaconda3\envs\ts_diffusion\lib\site-packages\torch\nn\modules\transformer.py:392: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.self_attn.batch_first was not True(use batch_first for better inference performance)
  warnings.warn(


# Feasibility Check: Training

In [15]:
import json
with open(f"../checkpoints/{ticker}_training_history_{date_to_load}.json", "r") as f:
    history = json.load(f)

In [16]:
# print(history)

In [17]:
history_df = pd.DataFrame(history)
history_df.describe()

,train_loss,val_loss,volatility_error
count,50.000000,50.0,50.000000
mean,0.302976,0.0,219.050572
std,0.037501,0.0,2.476320
min,0.248172,0.0,214.613351
25%,0.278448,0.0,216.462861
50%,0.302013,0.0,219.931588
75%,0.317433,0.0,221.045663
max,0.503335,0.0,223.212228


## Plotting and summarising info

In [18]:
if not execute_train and not load_outputs:
    print("Training skipped as per configuration.")
    model.load_state_dict(torch.load(checkpoint_path, map_location=device))
    model.eval()

    val_iter = iter(val_loader)
    test_batch = next(val_iter)

    with torch.no_grad():
        samples, observed, target_mask, obs_mask, obs_tp = model.evaluate(test_batch, n_samples=num_samples)

    print("Generated Samples shape:", samples.shape)


In [19]:
from datetime import datetime
today_str = datetime.now().strftime("%Y-%M-%d")

if not execute_train and not load_outputs:
    save_path = f"../outputs/{ticker}_{start_date}_{end_date}_{today_str}_csdi_out.pt"

    payload = {
        "samples": samples.detach().cpu(),
        "observed": observed.detach().cpu() if torch.is_tensor(observed) else observed,
        "target_mask": target_mask.detach().cpu(),
        "obs_mask": obs_mask.detach().cpu(),
        "obs_tp": obs_tp.detach().cpu(),
        "n_samples": num_samples,
    }

    torch.save(payload, save_path)
    print(f"Saved to {save_path}")


In [22]:
if load_outputs:
    # Define the path to your saved file
    load_path = f"../outputs/AAPL_2010-01-01_2024-12-31_2026-40-09_csdi_out.pt"

    # Load the dictionary
    # Note: Since you saved with .cpu(), these will load onto the CPU by default
    loaded_payload = torch.load(load_path, map_location='cpu')

    # Unpack the variables
    samples = loaded_payload["samples"]           # Shape: [n_samples, batch, features, length]
    observed = loaded_payload["observed"]         # The original ground truth data
    target_mask = loaded_payload["target_mask"]   # Mask for what was generated
    obs_mask = loaded_payload["obs_mask"]         # Mask for what was provided as condition
    obs_tp = loaded_payload["obs_tp"]             # Time points/indices
    num_samples = loaded_payload["n_samples"]     # The scalar value for count

    print(f"Loaded samples with shape: {samples.shape}")

Loaded samples with shape: torch.Size([8, 100, 5, 252])


In [23]:
# ==== Visualize sampled imputations (add this at the end of the notebook) ====
import os
import numpy as np
import matplotlib.pyplot as plt

# If your environment sometimes crashes on plt.show(), set this True to save PNGs instead.
SAVE_PLOTS = False
OUTDIR = "../checkpoints/figures"
os.makedirs(OUTDIR, exist_ok=True)

# ---- Safety checks: these must exist from your evaluate() cell ----
required = ["samples", "observed", "target_mask", "obs_tp"]
missing = [v for v in required if v not in globals()]
if missing:
    raise RuntimeError(
        f"Missing variables {missing}. Run the cell that calls model.evaluate(...) first."
    )

# ---- Pick which item in batch and which feature to visualize ----
b = 0  # batch index
K = samples.shape[2]
L = samples.shape[3]

# Adjust these names / index if your feature order differs:
feature_names = ["Open_rel", "High_rel", "Low_rel", "Volume_z", "Returns_z"]
if len(feature_names) != K:
    feature_names = [f"feat_{i}" for i in range(K)]

returns_idx = 4 if K >= 5 else 0  # change if needed

# ---- Move to CPU numpy ----
S = samples[b].detach().cpu().numpy()       # (n_samples, K, L)
X = observed[b].detach().cpu().numpy()      # (K, L)
TM = target_mask[b].detach().cpu().numpy()  # (K, L)

# time axis
tp = obs_tp
if tp.dim() == 2:
    t = tp[b].detach().cpu().numpy()
else:
    t = np.arange(L)

# ---- Compute summary statistics of samples ----
pred_mean = S.mean(axis=0)                        # (K, L)
pred_q10, pred_q90 = np.quantile(S, [0.1, 0.9], axis=0)  # (K, L) each

# ---- Metric on masked (target) points for the selected feature ----
mask_pos = TM[returns_idx].astype(bool)
if mask_pos.any():
    rmse = np.sqrt(np.mean((pred_mean[returns_idx, mask_pos] - X[returns_idx, mask_pos]) ** 2))
    mae = np.mean(np.abs(pred_mean[returns_idx, mask_pos] - X[returns_idx, mask_pos]))
    print(f"Imputation quality on masked points ({feature_names[returns_idx]}): RMSE={rmse:.6g}, MAE={mae:.6g}")
else:
    print(f"No masked target points found for {feature_names[returns_idx]} (target_mask is all zeros).")

# ---- Plot: true series vs sampled imputations (spaghetti + mean + 10–90 band) ----
n_show = min(20, S.shape[0])

plt.figure(figsize=(12, 4))
for i in range(n_show):
    plt.plot(t, S[i, returns_idx], alpha=0.15, linewidth=1)

plt.plot(t, pred_mean[returns_idx], linewidth=2, label="sample mean")
plt.fill_between(t, pred_q10[returns_idx], pred_q90[returns_idx], alpha=0.2, label="10–90% band")
plt.plot(t, X[returns_idx], linewidth=2, label="true (observed)")

if mask_pos.any():
    plt.scatter(t[mask_pos], X[returns_idx, mask_pos], s=25, label="masked target points")

plt.title(f"Sampled imputations for {feature_names[returns_idx]} (b={b}, n_samples={S.shape[0]})")
plt.xlabel("time")
plt.ylabel("value")
plt.grid(True, alpha=0.3)
plt.legend()

if SAVE_PLOTS:
    path = os.path.join(OUTDIR, f"samples_{feature_names[returns_idx]}_b{b}.png")
    plt.tight_layout()
    plt.savefig(path, dpi=150)
    plt.close()
    print("Saved:", path)
else:
    plt.show()

# ---- Optional: histogram of sampled values at one masked timestep ----
if mask_pos.any():
    idx0 = np.where(mask_pos)[0][0]  # first masked timestep
    vals = S[:, returns_idx, idx0]   # (n_samples,)

    plt.figure(figsize=(6, 3))
    plt.hist(vals, bins=30)
    plt.axvline(X[returns_idx, idx0], linewidth=2, label="true")
    plt.axvline(pred_mean[returns_idx, idx0], linewidth=2, label="mean")
    plt.title(f"Distribution of samples at t={idx0} ({feature_names[returns_idx]})")
    plt.xlabel("sampled value")
    plt.ylabel("count")
    plt.grid(True, alpha=0.3)
    plt.legend()

    if SAVE_PLOTS:
        path = os.path.join(OUTDIR, f"hist_{feature_names[returns_idx]}_b{b}_t{idx0}.png")
        plt.tight_layout()
        plt.savefig(path, dpi=150)
        plt.close()
        print("Saved:", path)
    else:
        plt.show()


No masked target points found for Returns_z (target_mask is all zeros).


: 